In [144]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)   

#funzione modello
def model1(x, par):
    x = x[0]
    A = par[0]
    B = par[1]
    return np.sqrt(1 + ((A*x**2)/(x**2 - B**2)))

def model2(x, par):
    x=x[0]
    A=par[0]
    B=par[1]
    return A*x + B

def model_derivative1(x, par):
    dx= 1e-7
    return (model1(x +dx, par) - model1(x, par))/dx

def model_derivative2(par):
    return  par[0]

#leggere file excel 
xlsx ="esperienza3.xlsx"
df = pd.read_excel(xlsx, sheet_name= "Dati")
df = df.dropna(subset=["lam", "n1", "err_lam", "err_n1"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
lam = df[:, 0]
n1 = df[:, 1]
err_lam = df[:, 2]
err_n1 = df[:, 3]


### FIT 1 

In [145]:
#minimizzazzione con ROOT
model_fit1 = ROOT.TF1('model1', model1, min(lam), max(lam), 2)
model_fit1.SetParameters(0.0, 0.0)
model_fit1.SetParNames("A", "B")

gr1 = ROOT.TGraphErrors(len(lam), lam, n1, err_lam, err_n1)
gr1.SetTitle("Dati sperimentali e fit; lambda; n'")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)

fit_res = gr1.Fit(model_fit1, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit1.GetParameters()

errors = model_fit1.GetParErrors()

chi2 = fit_res.Chi2()

err_n1eff = np.sqrt(err_n1**2 + ((model_derivative1(lam, par)) * err_lam)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print("------------------------------------------------------------------------")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")

leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit1, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

Fit status: 0
Parametri stimati:
A = 1.7374257753384694 ± 8.38200730039361e-06
B = 0.0 ± 1.4142135623730951
------------------------------------------------------------------------
Chi2 = 2296734.182716648
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  2.29673e+06
NDf                       =           12
Edm                       =   1.4145e-14
NCalls                    =           40
A                         =      1.73743   +/-   8.38201e-06 
B                         =            0   +/-   1.41421     


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [146]:
if np.abs(par[1]) < 3*errors[1]:
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")

B è compatibile con 0


In [147]:
n1_attesi = np.array([model_fit1.Eval(x_val) for x_val in lam])
residuals_norm = (n1 - n1_attesi) / err_n1eff

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm), max(residuals_norm), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "μ", "σ")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))

for val in residuals_norm:
    hist.Fill(val)
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"μ= {parameters[1]} ± {errors[1]}")
print(f"σ = {parameters[2]} ± {errors[2]}")
print(f"chi^2 = {chi2}")

c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")



Fit status: 0
Parametri stimati:
const = 9.915643394081497 ± 71.96998152629473
μ= 93.63786084377432 ± 2046.989843987537
σ = 96.71467573279342 ± 2109.669848756637
chi^2 = 1.0000003057403446
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =            1
NDf                       =            0
Edm                       =  3.05717e-07
NCalls                    =          112
const                     =      9.91564   +/-   71.97       
μ                        =      93.6379   +/-   2046.99     
σ                        =      96.7147   +/-   2109.67      	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [148]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print(f"const è compatibile con 1")
else :
    print(f"const NON è compatibile con 1")

print("---------------------------------------------------")

if np.abs(parameters[1]) < 3*errors[1]:
    print(f"μ è compatibile con 0")
else :
    print(f"μ NON è compatibile con 0")

print("--------------------------------------------------------")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print(f"σ è compatibile con 1")
else :
    print(f"σ NON è compatibile con 1")

const è compatibile con 1
---------------------------------------------------
μ è compatibile con 0
--------------------------------------------------------
σ è compatibile con 1


In [149]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = n1 - n1_attesi

line_fit = ROOT.TF1('line', line, min(lam), max(lam), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr4 = ROOT.TGraphErrors(len(lam), lam, residuals, np.zeros_like(residuals), err_n1eff)
gr4.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr4.SetMarkerStyle(20)
gr4.SetMarkerSize(0.7)
gr4.SetMarkerColor(ROOT.kAzure + 2)
gr4.SetLineColor(ROOT.kAzure + 3)

fit_res = gr4.Fit(line_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c4 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c4.SetGrid()

gr4.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr4, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c4.Draw()
c4.SaveAs("fit_line.png")

Fit status: 0
Parametri stimati:
m = -4.518598633335147e-05 ± 2.982032210772927e-08
q = 0.027326284821754143 ± 1.8210909331295308e-05
Chi2 = 676.0447234811188
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      676.045
NDf                       =           12
Edm                       =  3.74064e-08
NCalls                    =           40
m                         =  -4.5186e-05   +/-   2.98203e-08 
q                         =    0.0273263   +/-   1.82109e-05 


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created


In [150]:
if np.abs(parameters[0]) < 3*errors[0]:
    print(f"m è compatibile con 0")
else:
    print(f"m NON è compatibile con 0")

print("---------------------------------------------------")

if np.abs(parameters[1]) < 3*errors[1]:
    print(f"q è compatibile con 0")
else :
    print(f"q NON è compatibile con 0")

m NON è compatibile con 0
---------------------------------------------------
q NON è compatibile con 0


### FIT 2

In [151]:
#minimizzazzione con ROOT
model_fit2 = ROOT.TF1('model2', model2, min(lam), max(lam), 2)
model_fit2.SetParameters(0.0, 0.0)
model_fit2.SetParNames("A", "B")

gr2 = ROOT.TGraphErrors(len(lam), lam, n1, err_lam, err_n1)
gr2.SetTitle("Dati sperimentali e fit; lambda; n'")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)                    
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)        
gr2.SetLineWidth(1)

fit_res2 = gr2.Fit(model_fit2, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit2.GetParameters()

errors = model_fit2.GetParErrors()

chi2 = fit_res2.Chi2()

err_n1eff = np.sqrt(err_n1**2 + ((model_derivative2(lam)) * err_lam)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print("------------------------------------------------------------------------")
print(f"Chi2 = {chi2}")

c2 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c2.SetGrid()

gr2.Draw("AP")

leg2 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg2.AddEntry(gr2, "Dati", "pe")
leg2.AddEntry(model_fit2, "Fit", "l")
leg2.Draw()

c2.Draw()
c2.SaveAs("fit_root.png")

Fit status: 0
Parametri stimati:
A = -119.461255264066 ± 9.272690192098889
B = 65065.93864222683 ± 5050.396278767713
------------------------------------------------------------------------
Chi2 = 985.8910592238462
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      985.891
NDf                       =           12
Edm                       =  3.07142e-06
NCalls                    =          253
A                         =     -119.461   +/-   9.27269     
B                         =      65065.9   +/-   5050.4      


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [152]:
if np.abs(par[1]) < 3*errors[1]:
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")

B NON è compatibile con 0


In [153]:
n1_attesi = np.array([model_fit2.Eval(x_val) for x_val in lam])
residuals_norm = (n1 - n1_attesi) / err_n1eff

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm), max(residuals_norm), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "μ", "σ")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))

for val in residuals_norm:
    hist.Fill(val)
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "SR")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"μ= {parameters[1]} ± {errors[1]}")
print(f"σ = {parameters[2]} ± {errors[2]}")
print(f"chi^2 = {chi2}")

c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")

Fit status: 0
Parametri stimati:
const = 1.2039486297158735 ± 1.9396256234497034
μ= -10.56087639931862 ± 63.121936783425824
σ = 19.88732939520746 ± 17.46096053177822
chi^2 = 0.4496195036131376
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      0.44962
NDf                       =            9
Edm                       =  1.76673e-05
NCalls                    =          274
const                     =      1.20395   +/-   1.93963     
μ                        =     -10.5609   +/-   63.1219     
σ                        =      19.8873   +/-   17.461       	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [154]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print(f"const è compatibile con 1")
else :
    print(f"const NON è compatibile con 1")

print("---------------------------------------------------")

if np.abs(parameters[1]) < 3*errors[1]:
    print(f"μ è compatibile con 0")
else :
    print(f"μ NON è compatibile con 0")

print("--------------------------------------------------------")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print(f"σ è compatibile con 1")
else :
    print(f"σ NON è compatibile con 1")

const è compatibile con 1
---------------------------------------------------
μ è compatibile con 0
--------------------------------------------------------
σ è compatibile con 1


In [155]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = n1 - n1_attesi

line_fit = ROOT.TF1('line', line, min(lam), max(lam), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr4 = ROOT.TGraphErrors(len(lam), lam, residuals, np.zeros_like(residuals), err_n1eff)
gr4.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr4.SetMarkerStyle(20)
gr4.SetMarkerSize(0.7)
gr4.SetMarkerColor(ROOT.kAzure + 2)
gr4.SetLineColor(ROOT.kAzure + 3)

fit_res = gr4.Fit(line_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c4 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c4.SetGrid()

gr4.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c4.Draw()
c4.SaveAs("fit_line.png")

Fit status: 0
Parametri stimati:
m = 119.46118941188479 ± 14.33170337871994
q = -65064.24366849102 ± 7897.768967784538
Chi2 = 1.666389265833473e-12
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  1.66639e-12
NDf                       =           12
Edm                       =  7.72555e-18
NCalls                    =           37
m                         =      119.461   +/-   14.3317     
q                         =     -65064.2   +/-   7897.77     


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created


In [156]:
if np.abs(parameters[0]) < 3*errors[0]:
    print(f"m è compatibile con 0")
    print(f"m NON è compatibile con 0")

print("---------------------------------------------------")

if np.abs(parameters[1]) < 3*errors[1]:
    print(f"q è compatibile con 0")
else :
    print(f"q NON è compatibile con 0")


---------------------------------------------------
q NON è compatibile con 0
